# Project 1: Beginner KG Builder — Walkthrough

This notebook walks through the complete pipeline for building a knowledge graph from text documents.
We will extract entities, identify relationships, build a graph with NetworkX, visualize it, and run queries.

**Prerequisites:** Make sure you have run `pip install networkx matplotlib langchain langchain-openai pydantic` and have your API key configured.

In [ ]:
# Setup: imports and path configuration
import sys
import json
from pathlib import Path
from pydantic import BaseModel, Field

# Add projects dir to path for shared imports
sys.path.insert(0, str(Path(".").resolve().parent.parent))

from shared.llm_clients import chat_completion, chat_completion_structured
from shared.document_loader import load_text_files

import networkx as nx
import matplotlib.pyplot as plt

# Project paths
PROJECT_DIR = Path(".").resolve().parent
DATA_DIR = PROJECT_DIR / "data" / "sample_articles"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")
print("Setup complete!")

In [ ]:
# Step 1: Load sample documents
documents = load_text_files(str(DATA_DIR))

print(f"Loaded {len(documents)} documents:\n")
for doc in documents:
    print(f"  - {doc['filename']} ({len(doc['content'].split())} words)")
    print(f"    Preview: {doc['content'][:100]}...")
    print()

In [ ]:
# Step 2: Extract entities from each document

class Entity(BaseModel):
    name: str = Field(description="Name of the entity")
    entity_type: str = Field(description="Type: PERSON, ORGANIZATION, TECHNOLOGY, CONCEPT, LOCATION")
    description: str = Field(description="Brief description of the entity")

class EntityList(BaseModel):
    entities: list[Entity] = Field(description="List of extracted entities")

all_entities = []

for doc in documents:
    print(f"Extracting entities from: {doc['filename']}")
    result = chat_completion_structured(
        prompt=f"Extract all named entities from this text:\n\n{doc['content'][:2000]}",
        output_schema=EntityList,
        system="Extract entities with their types. Be thorough but precise.",
    )
    for entity in result.entities:
        entity_dict = entity.model_dump()
        entity_dict["source"] = doc["filename"]
        all_entities.append(entity_dict)
        print(f"  [{entity.entity_type}] {entity.name}")
    print()

print(f"\nTotal entities extracted: {len(all_entities)}")

In [ ]:
# Step 3: Extract relationships between entities

class Relationship(BaseModel):
    source: str = Field(description="Source entity name")
    target: str = Field(description="Target entity name")
    relation: str = Field(description="Type of relationship")
    description: str = Field(description="Brief description of the relationship")

class RelationshipList(BaseModel):
    relationships: list[Relationship] = Field(description="List of relationships")

all_relationships = []

for doc in documents:
    # Get entity names from this document
    doc_entities = [e["name"] for e in all_entities if e["source"] == doc["filename"]]
    entity_list = ", ".join(doc_entities)

    print(f"Extracting relationships from: {doc['filename']}")
    result = chat_completion_structured(
        prompt=f"""Given these entities: {entity_list}

And this text:
{doc['content'][:2000]}

Extract all relationships between the entities.""",
        output_schema=RelationshipList,
        system="Extract relationships between entities. Use concise relationship types like FOUNDED, WORKS_AT, DEVELOPED, USES, PART_OF.",
    )
    for rel in result.relationships:
        rel_dict = rel.model_dump()
        rel_dict["source_doc"] = doc["filename"]
        all_relationships.append(rel_dict)
        print(f"  {rel.source} --[{rel.relation}]--> {rel.target}")
    print()

print(f"\nTotal relationships extracted: {len(all_relationships)}")

In [ ]:
# Step 4: Build the knowledge graph with NetworkX

G = nx.DiGraph()

# Add entity nodes
for entity in all_entities:
    G.add_node(
        entity["name"],
        entity_type=entity["entity_type"],
        description=entity["description"],
    )

# Add relationship edges
for rel in all_relationships:
    if rel["source"] in G and rel["target"] in G:
        G.add_edge(
            rel["source"],
            rel["target"],
            relation=rel["relation"],
            description=rel["description"],
        )

print(f"Knowledge Graph built!")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Directed: {G.is_directed()}")

In [ ]:
# Step 5: Graph statistics

from collections import Counter

# Entity type distribution
type_counts = Counter(data.get("entity_type", "UNKNOWN") for _, data in G.nodes(data=True))
print("Entity types:")
for etype, count in type_counts.most_common():
    print(f"  {etype}: {count}")

# Relationship type distribution
rel_counts = Counter(data.get("relation", "UNKNOWN") for _, _, data in G.edges(data=True))
print("\nRelationship types:")
for rtype, count in rel_counts.most_common():
    print(f"  {rtype}: {count}")

# Most connected entities
print("\nTop 10 entities by degree:")
for node, degree in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node}: {degree} connections")

# Density
print(f"\nGraph density: {nx.density(G):.4f}")

In [ ]:
# Step 6: Visualize the knowledge graph

%matplotlib inline

# Color nodes by type
color_map = {
    "PERSON": "#4e79a7",
    "ORGANIZATION": "#f28e2b",
    "TECHNOLOGY": "#e15759",
    "CONCEPT": "#76b7b2",
    "LOCATION": "#59a14f",
}

node_colors = [
    color_map.get(G.nodes[n].get("entity_type", ""), "#bab0ac")
    for n in G.nodes()
]

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=600, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold", ax=ax)
nx.draw_networkx_edges(G, pos, edge_color="#999999", arrows=True, arrowsize=15, ax=ax)

# Edge labels
edge_labels = {(u, v): d.get("relation", "") for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6, font_color="#666666", ax=ax)

# Legend
for etype, color in color_map.items():
    ax.scatter([], [], c=color, s=100, label=etype)
ax.legend(loc="upper left", fontsize=9)

ax.set_title("Knowledge Graph Visualization", fontsize=16)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Query the graph

def find_paths(G, source, target, max_depth=3):
    """Find all paths between two entities."""
    try:
        paths = list(nx.all_simple_paths(G, source, target, cutoff=max_depth))
        return paths
    except nx.NetworkXError:
        return []

def get_entity_context(G, entity_name):
    """Get all information about an entity and its neighbors."""
    if entity_name not in G:
        return f"Entity '{entity_name}' not found in the graph."
    
    info = [f"Entity: {entity_name}"]
    info.append(f"Type: {G.nodes[entity_name].get('entity_type', 'unknown')}")
    info.append(f"Description: {G.nodes[entity_name].get('description', 'N/A')}")
    
    # Outgoing relationships
    for _, target, data in G.out_edges(entity_name, data=True):
        info.append(f"  --[{data.get('relation', '')}]--> {target}")
    
    # Incoming relationships
    for source, _, data in G.in_edges(entity_name, data=True):
        info.append(f"  <--[{data.get('relation', '')}]-- {source}")
    
    return "\n".join(info)

# Demo: show context for a few entities
for node in list(G.nodes())[:3]:
    print(get_entity_context(G, node))
    print()

In [ ]:
# Step 8: LLM-powered graph Q&A

def ask_graph(G, question):
    """Answer a question using graph context."""
    # Build context from graph
    triples = []
    for source, target, data in G.edges(data=True):
        relation = data.get("relation", "RELATED_TO")
        triples.append(f"({source}) --[{relation}]--> ({target})")
    
    context = "\n".join(triples)
    
    answer = chat_completion(
        prompt=f"""Based on this knowledge graph:

{context}

Question: {question}

Answer based only on the information in the graph.""",
        system="You are a knowledge graph analyst. Answer questions using only the provided graph data.",
    )
    return answer

# Example questions
questions = [
    "What are the main technologies mentioned and how are they related?",
    "What organizations are represented and what do they do?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_graph(G, q)}")
    print()

## Summary and Next Steps

In this walkthrough, we:

1. **Loaded** sample text documents
2. **Extracted entities** using an LLM with structured output
3. **Extracted relationships** between entities
4. **Built a knowledge graph** with NetworkX
5. **Computed statistics** about the graph structure
6. **Visualized** the graph with matplotlib
7. **Queried** the graph programmatically and with natural language

### Next Steps

- **Project 2 (Graph RAG)**: Use the graph to enhance retrieval-augmented generation
- **Project 3 (KG Agent)**: Build an agent that can reason over the graph
- Try with your own documents by placing them in `data/sample_articles/`
- Experiment with different LLM providers by changing the `provider` parameter